# PREPROCESSING

- [Importar librerías](#Importar-librerías)
- [Lectura de los datasets](#Lectura-de-los-datasets)
- [Eliminación de características irrelevantes](#Eliminación-de-características-irrelevantes)
- [Manejo de datos faltantes](#Manejo-de-datos-faltantes)
- [Manejo de outliers](#Manejo-de-outliers)
- [Gestión de tipos](#Gestión-de-tipos)
- [Codificación de variables categóricas](#Codificación-de-variables-categóricas)
- [Transformaciones de datos](#Transformaciones-de-datos)
- [Normalización y estandarización](#Normalización-y-estandarización)
- [Escritura de los dataframes resultantes](#Escritura-de-los-dataframes-resultantes)

## Importar librerías

In [1]:
# Entorno de la asignatura: proyecto_ml (kernel "Python (proyecto_ml)").
# Ya incluye todas las librerías de este notebook: no hace falta instalar nada.

In [2]:
import os
import shutil
import zipfile
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

## Lectura de los datasets

In [3]:
INPUT_ZIP = "./00_Data/Raw/titanic.zip"  # Directorio del zip
OUTPUT_FOLDER = "./00_Data/Raw"  # Directorio de destino
TRAIN_FILENAME = "train.csv"  # Nombre del fichero de entrenamiento
TEST_FILENAME = "test.csv"  # Nombre del fichero de entrenamiento

def fetch_data(input_path=INPUT_ZIP, output_dir=OUTPUT_FOLDER):
    """
    Extrae el contenido de un archivo ZIP en un directorio de destino.

    Parámetros:
    -----------
    input_path : str, opcional
        Ruta al archivo ZIP que se desea descomprimir. El valor predeterminado es la variable 'INPUT_ZIP'.
        
    output_dir : str, opcional
        Directorio en el cual se extraerá el contenido del archivo ZIP. Si el directorio no existe,
        será creado automáticamente. El valor predeterminado es la variable 'OUTPUT_FOLDER'.

    Comportamiento:
    ---------------
    - Crea el directorio de destino si no existe.
    - Descomprime el archivo ZIP en el directorio de destino.

    Excepciones:
    ------------
    Puede lanzar una excepción si el archivo ZIP no existe o si hay problemas al descomprimirlo.

    Ejemplo de uso:
    ---------------
    fetch_data('data.zip', 'output/')
    """
    # Comprobación de que el directorio de destino existe
    os.makedirs(output_dir, exist_ok=True)

    # Descomprime el archivo ZIP en caso de que no haya ningún csv en la carpeta
    if(len([file for file in os.listdir(output_dir) if file.endswith('.csv')]) == 0):
        with zipfile.ZipFile(input_path, 'r') as zip_ref:
            zip_ref.extractall(output_dir)


def load_data(directory=OUTPUT_FOLDER, filename=TRAIN_FILENAME):
    """
    Lee un archivo CSV desde el directorio especificado.

    Parámetros:
    -----------
    directory : str
        El directorio donde se encuentra el archivo CSV.
        
    filename : str
        El nombre del archivo CSV a leer (incluyendo la extensión .csv).

    Retorna:
    --------
    pd.DataFrame
        Un DataFrame de pandas que contiene los datos del archivo CSV.

    Excepciones:
    ------------
    FileNotFoundError:
        Se lanza si el archivo no existe en el directorio dado.
    
    Ejemplo de uso:
    ---------------
    df = read_csv_from_directory('data', 'file.csv')
    """
    # Construir la ruta completa al archivo CSV
    file_path = os.path.join(directory, filename)

    # Verificar si el archivo existe
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"El archivo {filename} no se encuentra en el directorio {directory}")

    # Leer el archivo CSV en un DataFrame
    return pd.read_csv(file_path)

fetch_data()
df_train = load_data(OUTPUT_FOLDER, TRAIN_FILENAME)
df_test = load_data(OUTPUT_FOLDER, TEST_FILENAME)

print("Train dataset:", df_train.shape)
print("Test dataset:", df_test.shape)

Train dataset: (891, 12)
Test dataset: (418, 11)


Como vemos, en el dataset de test no se incluyen las 12 columnas (quitan la de predicción)

## Eliminación de características irrelevantes

In [4]:
# Seleccionamos las columnas con las que nos queremos quedar
# (mantenemos Name para extraer el título, que el EDA mostró como el predictor más fuerte)
cols_keep = ["Sex", "Pclass", "Age", "SibSp", "Parch", "Embarked", "Cabin", "Fare", "Name"]
predict_col = "Survived"

In [5]:
print("Las columnas originales son:", df_train.columns.tolist())

df_train = df_train[cols_keep + [predict_col]]
df_test = df_test[cols_keep]

print("Las columnas tras eliminar irrelevantes son:", df_train.columns.tolist())

Las columnas originales son: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Las columnas tras eliminar irrelevantes son: ['Sex', 'Pclass', 'Age', 'SibSp', 'Parch', 'Embarked', 'Cabin', 'Fare', 'Name', 'Survived']


## Extracción de `Title` desde el nombre

El nombre por sí solo no sirve, pero el **título** (Mr, Mrs, Miss, Master...) resume sexo, edad y estatus en una sola variable. En el EDA fue el predictor más fuerte. Agrupamos los títulos raros en `Rare` para no crear categorías con muy pocos casos.

In [6]:
import re

def extraer_title(nombre):
    m = re.search(r',\s*([^\.]+)\.', nombre)
    t = m.group(1).strip() if m else 'None'
    # unificamos variantes equivalentes
    t = {'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'}.get(t, t)
    # el resto (Dr, Rev, Col, Major, Lady...) van a 'Rare'
    if t not in ['Mr', 'Mrs', 'Miss', 'Master']:
        t = 'Rare'
    return t

df_train['Title'] = df_train['Name'].apply(extraer_title)
df_test['Title']  = df_test['Name'].apply(extraer_title)

# ya no necesitamos el nombre completo
df_train = df_train.drop(columns='Name')
df_test  = df_test.drop(columns='Name')

df_train['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

## Manejo de datos faltantes

In [7]:
df_train.isnull().sum()

Sex           0
Pclass        0
Age         177
SibSp         0
Parch         0
Embarked      2
Cabin       687
Fare          0
Survived      0
Title         0
dtype: int64

In [8]:
df_test.isnull().sum()

Sex           0
Pclass        0
Age          86
SibSp         0
Parch         0
Embarked      0
Cabin       327
Fare          1
Title         0
dtype: int64

In [9]:
# Definimos función para imputación simple
def simple_imputation(col_name, training, test, strategy='median'):
    """
    Realiza una imputación simple de valores faltantes en una columna específica de los 
    conjuntos de entrenamiento y test, utilizando la estrategia especificada.

    Args:
        strategy (str, opcional): La estrategia de imputación a utilizar. Puede ser 'mean', 
                                  'median', 'most_frequent', o 'constant'. Por defecto es 'median'.
        col_name (str): El nombre de la columna sobre la cual se realizará la imputación.
        training (pd.DataFrame): El conjunto de datos de entrenamiento que contiene la columna con valores faltantes.
        test (pd.DataFrame): El conjunto de datos de prueba que contiene la columna con valores faltantes.

    Returns:
        tuple: Una tupla que contiene los conjuntos de datos de entrenamiento y prueba después de aplicar la imputación.

    Raises:
        ValueError: Si el nombre de la columna no existe en los DataFrames proporcionados.
    """

    # Definición del imputador
    imputer = SimpleImputer(strategy=strategy)

    # Entrenamiento del imputer + transformación en training
    training[col_name] = imputer.fit_transform(training[[col_name]]).ravel()  # ravel para aplanar la matriz 2D

    # Transformación en test
    test[col_name] = imputer.transform(test[[col_name]]).ravel()  # ravel para aplanar la matriz 2D

    return training, test

In [10]:
# Imputamos 'Age' con la mediana DENTRO de cada Title (Master≈niño, Mrs≈adulta...),
# mucho más precisa que una mediana global. Las medianas se calculan SOLO con train.
medianas_edad = df_train.groupby('Title')['Age'].median()
mediana_global = df_train['Age'].median()

def imputa_edad_por_title(df):
    faltan = df['Age'].isnull()
    df.loc[faltan, 'Age'] = df.loc[faltan, 'Title'].map(medianas_edad).fillna(mediana_global)
    return df

df_train = imputa_edad_por_title(df_train)
df_test  = imputa_edad_por_title(df_test)

In [11]:
# Con cabin, puesto que creamos variable auxiliar de 'Has_Cabin', lo hacemos aquí y eliminamos la original
df_train['Has_Cabin'] = df_train['Cabin'].notnull().astype(int)
df_train = df_train.drop('Cabin', axis=1)
df_test['Has_Cabin'] = df_test['Cabin'].notnull().astype(int)
df_test = df_test.drop('Cabin', axis=1)

In [12]:
# Imputamos en columna 'Embarked' con la moda (categórica)
df_train, df_test = simple_imputation('Embarked', df_train, df_test, 'most_frequent')

In [13]:
# Al detectar que tenemos un missing value en test para 'Fare', lo imputamos con la mediana de training
_, df_test = simple_imputation('Fare', df_train, df_test)

In [14]:
# Check para ver si hemos imputado correctamente en train
df_train.isnull().sum()

Sex          0
Pclass       0
Age          0
SibSp        0
Parch        0
Embarked     0
Fare         0
Survived     0
Title        0
Has_Cabin    0
dtype: int64

In [15]:
# Check para ver si hemos imputado correctamente en test
df_test.isnull().sum()

Sex          0
Pclass       0
Age          0
SibSp        0
Parch        0
Embarked     0
Fare         0
Title        0
Has_Cabin    0
dtype: int64

## Manejo de outliers

En este caso hemos estudiado los outliers y parece que son desviaciones estadísticas, nada que necesitemos eliminar o imputar para nuestro problema.

## Gestión de tipos

Mantenemos `Age` como variable **numérica continua**: los modelos de árbol encuentran sus propios cortes y no conviene tirar información binándola. La convertimos a entero (los años son enteros).

In [16]:
df_train['Age'] = df_train['Age'].round().astype(int)
df_test['Age']  = df_test['Age'].round().astype(int)

## Codificación de variables categóricas

In [17]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

def one_hot_encode_column(col_name, training, test):
    """
    Aplica One-Hot Encoding a una columna categórica en los DataFrames de entrenamiento y prueba.
    
    La función codifica la columna indicada en ambos DataFrames utilizando One-Hot Encoding
    con el encoder de scikit-learn. Asegura que ambos DataFrames tengan las mismas columnas 
    después de la codificación. Además, elimina una columna codificada para evitar multicolinealidad,
    y convierte los valores resultantes a enteros.

    Args:
        col_name (str): El nombre de la columna categórica a codificar.
        training (pd.DataFrame): El DataFrame de entrenamiento.
        test (pd.DataFrame): El DataFrame de prueba.

    Returns:
        tuple: Una tupla que contiene dos DataFrames (entrenamiento y prueba) 
               con la columna especificada codificada mediante One-Hot Encoding.
    """
    
    # Inicializar OneHotEncoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    
    # Ajustar el encoder con el DataFrame de entrenamiento y transformar
    training_encoded = encoder.fit_transform(training[[col_name]])
    test_encoded = encoder.transform(test[[col_name]])  # Usar transform para test sin ajustar de nuevo
    
    # Convertir los resultados codificados a DataFrames con los nombres de columnas adecuados
    training_encoded_df = pd.DataFrame(training_encoded, 
                                       columns=encoder.get_feature_names_out([col_name]),
                                       index=training.index)
    
    test_encoded_df = pd.DataFrame(test_encoded, 
                                   columns=encoder.get_feature_names_out([col_name]),
                                   index=test.index)
    
    # Convertir los valores codificados a enteros
    training_encoded_df = training_encoded_df.astype(int)
    test_encoded_df = test_encoded_df.astype(int)
    
    # Eliminar una columna codificada para evitar multicolinealidad (por ejemplo, la primera columna)
    training_encoded_df = training_encoded_df.drop(columns=encoder.get_feature_names_out([col_name])[0])
    test_encoded_df = test_encoded_df.drop(columns=encoder.get_feature_names_out([col_name])[0])
    
    # Combinar los DataFrames codificados con los DataFrames originales, excluyendo la columna original
    training_combined = pd.concat([training.drop(columns=[col_name]), training_encoded_df], axis=1)
    test_combined = pd.concat([test.drop(columns=[col_name]), test_encoded_df], axis=1)
    
    # Retornar los DataFrames de entrenamiento y prueba codificados
    return training_combined, test_combined


In [18]:
# Codificación de 'Sex' (one-hot)
df_train, df_test = one_hot_encode_column('Sex', df_train, df_test)

In [19]:
# Codificación de 'Embarked' (one-hot)
df_train, df_test = one_hot_encode_column('Embarked', df_train, df_test)

In [20]:
# Codificación de 'Title' (one-hot)
df_train, df_test = one_hot_encode_column('Title', df_train, df_test)

## Transformaciones de datos

In [21]:
# 'Fare' está muy sesgada (cola larga a la derecha). log1p comprime esa cola
# y ayuda mucho a los modelos lineales y a las redes neuronales.
df_train['Fare_log'] = np.log1p(df_train['Fare'])
df_test['Fare_log']  = np.log1p(df_test['Fare'])
df_train = df_train.drop(columns='Fare')
df_test  = df_test.drop(columns='Fare')

In [22]:
# Creación de 'FamilySize' y de 'IsAlone'
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch'] + 1
df_test['FamilySize']  = df_test['SibSp'] + df_test['Parch'] + 1

# El EDA mostró que viajar solo reduce la supervivencia y que las familias
# medianas (2-4) sobreviven más: 'IsAlone' captura ese punto.
df_train['IsAlone'] = (df_train['FamilySize'] == 1).astype(int)
df_test['IsAlone']  = (df_test['FamilySize'] == 1).astype(int)

df_train = df_train.drop(columns=['SibSp', 'Parch'])
df_test  = df_test.drop(columns=['SibSp', 'Parch'])

In [23]:
df_train

,Pclass,Age,Survived,Has_Cabin,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Fare_log,FamilySize,IsAlone
0,3,22,0,0,1,0,1,0,1,0,0,2.110213,2,0
1,1,38,1,1,0,0,0,0,0,1,0,4.280593,2,0
2,3,26,1,0,0,0,1,1,0,0,0,2.188856,1,1
3,1,35,1,1,0,0,1,0,0,1,0,3.990834,2,0
4,3,35,0,0,1,0,1,0,1,0,0,2.202765,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,2,27,0,0,1,0,1,0,0,0,1,2.639057,1,1
887,1,19,1,1,0,0,1,1,0,0,0,3.433987,1,1
888,3,21,0,0,0,0,1,1,0,0,0,3.196630,4,0
889,1,26,1,1,1,0,0,0,1,0,0,3.433987,1,1


In [24]:
df_test

,Pclass,Age,Has_Cabin,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Fare_log,FamilySize,IsAlone
0,3,34,0,1,1,0,0,1,0,0,2.178064,1,1
1,3,47,0,0,0,1,0,0,1,0,2.079442,2,0
2,2,62,0,1,1,0,0,1,0,0,2.369075,1,1
3,3,27,0,1,0,1,0,1,0,0,2.268252,1,1
4,3,22,0,0,0,1,0,0,1,0,2.586824,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,3,30,0,1,0,1,0,1,0,0,2.202765,1,1
414,1,39,1,0,0,0,0,0,0,1,4.699571,1,1
415,3,38,0,1,0,1,0,1,0,0,2.110213,1,1
416,3,30,0,1,0,1,0,1,0,0,2.202765,1,1


## Normalización y estandarización
Tenemos variables numéricas en escalas muy distintas: `Age` va de 0 a 80, `FamilySize` de 1 a 11, `Fare_log` de 0 a 6 y `Pclass` de 1 a 3. Los modelos basados en distancias o en coeficientes (Regresión Logística, SVM, redes neuronales) son sensibles a la escala: sin estandarizar, la variable con números más grandes domina. Los árboles no se ven afectados, así que estandarizar no les perjudica.

Estandarizamos con `StandardScaler` (media 0 y desviación 1) **solo las variables numéricas no binarias**. Las binarias (0/1) las dejamos tal cual: ya están en una escala comparable y así siguen siendo fáciles de interpretar.

El escalador se **ajusta solo con `train`** (de ahí salen la media y la desviación) y se **aplica igual al `test` de Kaggle**, que no participa en el cálculo. Es la misma regla que con la imputación de la edad: el test nunca aporta información al preprocesado.

In [25]:
from sklearn.preprocessing import StandardScaler

# Variables numéricas no binarias (las 0/1 se quedan como están)
cols_escalar = ['Pclass', 'Age', 'Fare_log', 'FamilySize']

scaler = StandardScaler()
df_train[cols_escalar] = scaler.fit_transform(df_train[cols_escalar])  # media y desviación SOLO de train
df_test[cols_escalar] = scaler.transform(df_test[cols_escalar])        # el test de Kaggle se transforma con las de train

# Comprobación: en train, media ≈ 0 y desviación ≈ 1
df_train[cols_escalar].describe().round(2)

,Pclass,Age,Fare_log,FamilySize
count,891.00,891.00,891.00,891.00
mean,-0.00,0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00
min,-1.57,-2.22,-3.06,-0.56
25%,-0.37,-0.63,-0.80,-0.56
50%,0.83,0.05,-0.23,-0.56
75%,0.83,0.42,0.52,0.06
max,0.83,3.82,3.39,5.64


## Escritura de los dataframes resultantes

In [26]:
OUTPUT_FOLDER = "./00_Data/Cleaned/"

def save_dataframes_to_csv(output_folder, df_train, df_test, train_filename="train_clean.csv", test_filename="test_clean.csv"):
    """
    Guarda los DataFrames de entrenamiento y prueba en formato CSV en una carpeta específica.
    Si la carpeta ya existe, borra todo su contenido antes de guardar los nuevos archivos.
    
    Args:
        output_folder (str): La ruta de la carpeta donde se guardarán los archivos CSV.
        df_train (pd.DataFrame): El DataFrame de entrenamiento que se va a guardar.
        df_test (pd.DataFrame): El DataFrame de prueba que se va a guardar.
        train_filename (str, opcional): El nombre del archivo CSV para el DataFrame de entrenamiento.
        test_filename (str, opcional): El nombre del archivo CSV para el DataFrame de prueba.
    
    """
    # Si la carpeta ya existe, eliminar todo su contenido
    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)  # Borrar toda la carpeta y su contenido
        print(f"Carpeta {output_folder} eliminada.")
    
    # Crear la carpeta si no existe
    os.makedirs(output_folder, exist_ok=True)
    
    # Definir las rutas completas de los archivos
    train_path = os.path.join(output_folder, train_filename)
    test_path = os.path.join(output_folder, test_filename)
    
    # Guardar los DataFrames en formato CSV
    df_train.to_csv(train_path, index=False)
    df_test.to_csv(test_path, index=False)
    
    print(f"DataFrames guardados en {output_folder}:")
    print(f" - {train_filename}")
    print(f" - {test_filename}")

save_dataframes_to_csv(OUTPUT_FOLDER, df_train, df_test)

Carpeta ./00_Data/Cleaned/ eliminada.
DataFrames guardados en ./00_Data/Cleaned/:
 - train_clean.csv
 - test_clean.csv
